Embedding lookup: from tokens to vectors

How nn.Embedding works as a learnable lookup table with similarity computation

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

vocab_size = 10000 # number of unique tokens
embed_dim = 256 # embedding vector size

# creating embedding layer (stores a [10000,256] matrix)
embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

print(f'Embedding matrix shape: {embedding.weight.shape}')
print(f'Total parameters: {embedding.weight.numel():,}')

# simulate the tokens for - the cat sat on the mat
token_ids = torch.tensor([42, 1587, 923, 15, 42, 2041])

# Lookup: each index selects a row from the matrix
vectors = embedding(token_ids)
print(f'\nInput Shape: {token_ids.shape}')
print(f'\nOutput Shape: {vectors.shape}')

# same token will get same vector always - eg "the" and "The"
print(f'Same token same vector: {torch.equal(vectors[0], vectors[4])}')

# why it works: equivalent to one hot x matrix
# one hot approach (wasteful but mathematically correct)

one_hot = F.one_hot(torch.tensor(42), num_classes = vocab_size).float()
manual_lookup = one_hot @ embedding.weight # matrix multiply

# direct indexing
direct_lookup = embedding.weight[42]

print(f'\nOne-hot multiply == direct index: '
      f'{torch.allclose(manual_lookup,direct_lookup)}')


# similarity: trained embeddings cluster similar words
# after training similar words have high cosine similarity
v1 = vectors[0] # the
v2 = vectors[1] # cat
cosine_sim = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))
print(f'\nCosine Similarity (untrained, random): {cosine_sim.item():.4f}')
print("After training, similar words would score > 0.7")

v4 = vectors[4] # cat
cosine_sim = F.cosine_similarity(v1.unsqueeze(0), v4.unsqueeze(0))
print(f'\nCosine Similarity: {cosine_sim.item():.4f}')
print("After training, similar words would score > 0.7")

Embedding matrix shape: torch.Size([10000, 256])
Total parameters: 2,560,000

Input Shape: torch.Size([6])

Output Shape: torch.Size([6, 256])
Same token same vector: True

One-hot multiply == direct index: True

Cosine Similarity (untrained, random): -0.0560
After training, similar words would score > 0.7

Cosine Similarity: 1.0000
After training, similar words would score > 0.7


Vision Transformer patch embedding

Convert images into patch sequences for ViT, with positional encoding

In [15]:
import torch
import torch.nn as nn

# vit patch embedding: images as token sequences
class PatchEmbedding(nn.Module):
  """Convert image into a sequence of patch embeddings"""
  def __init__(self, img_size=224,patch_size=16,in_channels=3,embed_dim=768):
    super().__init__()
    self.patch_size = patch_size
    self.num_patches = (img_size // patch_size) ** 2 # 196

    # Conv2d with kernel=stride=patch_size acts as patch extractor
    # Each filter extracts one dimension from each patch
    self.projection = nn.Conv2d(
        in_channels, embed_dim, kernel_size=patch_size,stride=patch_size
    )

    # [CLS] token for classification (learnable)
    self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))

    # positional encoding (learnable, one per batch + cls)
    self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))

  def forward(self, x):
    B = x.shape[0] # batch size

    # Step 1: Extract patches via convulation
    # [B, 3, 224, 224] -> [B,768, 14, 14]
    patches = self.projection(x)

    # steps 2: flatten spatial dims to sequences
    # [B, 768, 14, 14] -> [B, 768, 196] -> [B, 196, 768]
    patches = patches.flatten(2).transpose(1,2)

    # step 3: prepend cls token
    cls = self.cls_token.expand(B, -1, -1) # [B, 1, 768]
    tokens = torch.cat([cls, patches], dim=1) # [B, 197, 768]

    # step 4: add position encoding
    tokens = tokens + self.pos_embedding # [B, 197, 768]

    return tokens


# demo
patch_embed = PatchEmbedding()
image = torch.randn(2,3,224,224) # 2 rgb image
tokens = patch_embed(image)

print(f"Image shape:    {image.shape}")     # [2, 3, 224, 224]
print(f"Token sequence: {tokens.shape}")    # [2, 197, 768]
print(f"Num patches:    {patch_embed.num_patches}")  # 196
print(f"Sequence = {patch_embed.num_patches} patches + 1 [CLS]")

# parameters breakdown
total = sum(p.numel() for p in patch_embed.parameters())
proj = sum(p.numel() for p in patch_embed.projection.parameters())
print(f"\nProjection params: {proj:,}")      # 3*16*16*768 + 768
print(f"CLS token:         {patch_embed.cls_token.numel():,}")
print(f"Pos embedding:     {patch_embed.pos_embedding.numel():,}")
print(f"Total:             {total:,}")

Image shape:    torch.Size([2, 3, 224, 224])
Token sequence: torch.Size([2, 197, 768])
Num patches:    196
Sequence = 196 patches + 1 [CLS]

Projection params: 590,592
CLS token:         768
Pos embedding:     151,296
Total:             742,656
